# Re-evaluate Model 2 (conditions) t5-small on the leak-free test set (Kaggle GPU)

`07_eval_conditions_t5small_compare.ipynb` scored both checkpoints on the 300k pool's
`conditions_test.jsonl` (7,714 rows) and called it leak-free. It is leak-free for the
**300k-trained** checkpoint only. The pools were sampled by independent runs and the
conditions split is drawn inside each pool, so **2,027 of those 7,714 rows are training
rows of the 41k-trained checkpoint** — identical on `product_smiles` and on the canonical
(product, reactants) pair alike.

This notebook re-scores both checkpoints on `data/v2_ord_conditions_test_clean.jsonl`
(5,687 rows), produced by `scripts/build_clean_conditions_test.py`, which neither
checkpoint was trained on. Rows carrying a ground-truth value, i.e. the denominators of
the per-field metrics: solvent 5,077 · catalyst 941 · temperature 1,579 · yield 3,440.

Inputs to attach, same as notebook 07: the 300k training kernel's output (`t5small-300k-ckpt`)
and the uploaded 41k checkpoint (`t5small-41k`).

below is what carries it onto Kaggle.
Inputs to attach: the two checkpoint datasets from notebook 07 plus
`kuzmenkooleh/retro-planner-ord-conditions-test-clean`, which carries the 5,687-row test set.

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, os, shutil
print("/kaggle/input contents:", os.listdir("/kaggle/input"))
cfgs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("config.json found:", *cfgs, sep="\n  ")

# The leak-free test set rides in as its own private dataset, so this notebook
# needs nothing committed to the repo beyond the eval script itself.
test_file = next(glob.iglob(
    "/kaggle/input/**/v2_ord_conditions_test_clean.jsonl", recursive=True))
new_src = os.path.dirname(next(c for c in cfgs if "t5small-300k-ckpt" in c))
old_src = os.path.dirname(next(c for c in cfgs if "t5small-41k" in c))

# /kaggle/input is read-only; the eval script may rewrite tokenizer/config in place,
# so copy each checkpoint to a writable working dir first.
new_dir = "/kaggle/working/new_model"; shutil.rmtree(new_dir, ignore_errors=True); shutil.copytree(new_src, new_dir)
old_dir = "/kaggle/working/old_model"; shutil.rmtree(old_dir, ignore_errors=True); shutil.copytree(old_src, old_dir)
print("test:", test_file, "rows", sum(1 for _ in open(test_file)))
print("NEW:", new_dir, "(from", new_src + ")")
print("OLD:", old_dir, "(from", old_src + ")")

In [ ]:
# NEW (300k-trained) on the leak-free test set
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{new_dir}" --test-file "{test_file}" \
    --num-beams 10 --device cuda --batch-size 64 \
    --output /kaggle/working/eval_new_300k_clean.json
print("NEW done")

In [ ]:
# OLD (41k-trained) on the SAME leak-free test set
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{old_dir}" --test-file "{test_file}" \
    --num-beams 10 --device cuda --batch-size 64 \
    --output /kaggle/working/eval_old_41k_clean.json
print("OLD done")

In [ ]:
import json
for tag, f in [("NEW 300k", "/kaggle/working/eval_new_300k_clean.json"),
               ("OLD 41k", "/kaggle/working/eval_old_41k_clean.json")]:
    print("===", tag, "===")
    print(json.dumps(json.load(open(f))["summary"], indent=2))